# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library, referencing all entities by their `@id` fields following the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review the available record sets along with their `@id`s and associated fields, all by unique identifier.

In [ ]:
# List all record sets by @id
record_sets = list(dataset.record_sets)
print(f"Record sets in dataset ({len(record_sets)} found):")
for rs in record_sets:
    print(f"  @id: {rs.id}, name: {rs.name}")

if len(record_sets) > 0:
    # Display fields for the first record set
    rs = record_sets[0]
    print(f"\nFields in Record Set '{rs.name}' (@id: {rs.id}):")
    for field in rs.fields:
        print(f"  Field @id: {field.id}, name: {field.name}, dataType: {getattr(field, 'data_type', None)}")
else:
    print("No record sets found in dataset.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis, referencing each record set and field by their `@id`.

In [ ]:
# Extract records from all record sets
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set: {record_set_id}")
    else:
        print(f"No records for record set: {record_set_id}")

# Show field (column) @id's of first dataframe if available
if len(dataframes) > 0:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns in DataFrame for {first_rs_id}:\n{dataframes[first_rs_id].columns.tolist()}")
    display(dataframes[first_rs_id].head())
else:
    print("No DataFrames created from record sets (none with records found).")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing operations: filtering, normalizing numeric fields, and grouping by key attributes, referencing columns by their `@id`.

For this example, the first record set (if available) is used for demonstration.

In [ ]:
# Proceed only if we have at least one DataFrame
if len(dataframes) > 0:
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]
    print(f"Working on record set: {rs_id}")

    # Identify numeric field(s) by looking for number-like columns
    numeric_columns = df.select_dtypes(include=["float", "int"]).columns.tolist()
    if numeric_columns:
        numeric_field_id = numeric_columns[0]
        print(f"\nNumeric field selected (by @id): {numeric_field_id}")

        # Filtering: choose a threshold based on sample min/max
        threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}")
        display(filtered_df.head(3))

        # Normalization
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head(3))

        # Grouping by a non-numeric column if available
        non_num_cols = [c for c in df.columns if c != numeric_field_id and df[c].dtype == object]
        group_field_id = non_num_cols[0] if non_num_cols else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().rename(columns={numeric_field_id:'mean_'+numeric_field_id})
            print(f"\nGrouped data by {group_field_id}:")
            display(grouped_df.head(3))
        else:
            print("No suitable non-numeric column for grouping found.")
    else:
        print("No numeric fields found in the record set.")
else:
    print("No dataframes loaded for EDA.")

## 5. Visualization
Visualize numeric field distribution and grouping, referencing column names by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only proceed if EDA code above has produced data
if len(dataframes) > 0 and 'filtered_df' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If group_field_id and grouped_df exist, plot group mean
    if 'group_field_id' in locals() and group_field_id and 'grouped_df' in locals():
        plt.figure(figsize=(8,4))
        sns.barplot(data=grouped_df, x=group_field_id, y='mean_'+numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
This notebook demonstrated data exploration steps using the `mlcroissant` library on the FAIR^2 dataset. All entities, fields, and columns were referenced by their unique `@id`, following best practices for Croissant schema usage.

Key findings involve:
- Listing and understanding record sets and their fields by identifier.
- Extracting and loading dataset slices into DataFrames by `@id`.
- Performing filtering, normalization, grouping operations, and visualizing the results.

Explore further to analyze predictors of knowledge adoption and rangeland management practices in Northern Kenya using this FAIR and transparent workflow.